# Smart Airport Passenger Assistance Multimodal Chatbot
**MSc in Artificial Intelligence — Multi-Modal Chatbots (Set exercise)**

End-to-end notebook: environment setup → data acquisition & exploration → preprocessing →
model training (vision CNN + FAISS, domain-adapted speech-to-text, intent/retrieval NLP) →
multimodal fusion → evaluation → interface.

> Run cells top-to-bottom. On a fresh machine (or Colab) the whole pipeline takes ~10–15 min on CPU.
> The Streamlit interface (Task 6) is launched separately with `streamlit run app/app.py`.

## 1. Environment setup (Task 1)

In [ ]:
# Install dependencies (uncomment on a fresh machine / Colab)
# !pip install -r requirements.txt
# !python -m spacy download en_core_web_sm
# !apt-get install -y espeak-ng ffmpeg     # needed only to (re)generate the voice dataset

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import torch, faiss, spacy, librosa, sklearn, pocketsphinx, streamlit
print("torch", torch.__version__, "| faiss OK | spaCy", spacy.__version__,
      "| librosa", librosa.__version__, "| sklearn", sklearn.__version__)

## 2. Data acquisition (Task 2)
Generates the 540-image synthetic signage dataset, the 112-query labelled text dataset, the 66-file voice dataset, and produces the exploration figures. The 18-record airport knowledge base is in `kb/airport_kb.json`.

In [ ]:
!python src/generate_images.py
!python src/generate_queries.py

In [ ]:
!python src/explore_data.py

In [ ]:
from IPython.display import Image as IPyImage, display
for f in ["sample_grid.png", "class_distribution.png", "similar_dissimilar.png",
          "intent_distribution.png", "audio_example.png"]:
    display(IPyImage(filename=f"outputs/figures/{f}", width=650))

In [ ]:
import json
kb = json.load(open("kb/airport_kb.json"))
print(f"Knowledge base: {len(kb['records'])} records")
kb["records"][0]   # example record + schema

## 3. Preprocessing pipelines (Task 3)
Defined in `src/preprocessing.py`: image (load → resize → augment → tensor → normalise), audio (16 kHz mono → trim → pad + dither → STT), text (clean → tokenise → entities → TF-IDF), shared by typed and transcribed queries.

In [ ]:
from preprocessing import make_loaders, extract_entities, clean_text, tokenise
tr, va, te, classes = make_loaders()
xb, yb = next(iter(tr))
print("batch:", xb.shape, "| classes:", classes)
print(clean_text("Where is GATE B12?!"), "->", tokenise("Where is gate B12?"))
print("entities:", extract_entities("Where are check in desks 30 to 40 in terminal 1?"))

## 4–5. Model training and evaluation (Tasks 4–5)
Trains the sign CNN (60 epochs, CPU-friendly), builds the FAISS index, trains the intent classifier, then runs the full evaluation: vision top-1/top-3 + stress set, speech WER (domain-adapted vs open dictation), intent P/R/F1 + confusion matrix, KB retrieval accuracy, and the 7 multimodal fusion scenarios.

In [ ]:
%cd src
!python vision.py
!python nlp.py
!python evaluate.py
%cd ..

In [ ]:
for f in ["training_curves.png", "vision_examples.png", "confusion_matrix.png",
          "augmentation_examples.png"]:
    display(IPyImage(filename=f"outputs/figures/{f}", width=650))

In [ ]:
import json
for name in ["vision_metrics", "speech_metrics", "retrieval_metrics"]:
    m = json.load(open(f"outputs/eval/{name}.json"))
    m.pop("rows", None); m.pop("test_rows", None); m.pop("stress_rows", None)
    print(name, "->", json.dumps(m, indent=1)[:400], "\n")

## Multimodal fusion demo (Task 4.4)
Ask the assistant with text, an image, voice, or combinations.

In [ ]:
from fusion import AirportAssistant
from PIL import Image
bot = AirportAssistant()

r = bot.respond(text="Where is gate B12?")
print(r["message"][:200], "\nconfidence:", r["confidence"])

In [ ]:
r = bot.respond(image=Image.open("data/images/baggage_claim/baggage_claim_003.jpg"))
print(r["message"][:150], "\nconfidence:", r["confidence"], "| vision:", r["details"]["vision"]["category"])

In [ ]:
r = bot.respond(audio_path="data/audio/q000.wav")
print("transcript:", r["details"]["transcript"])
print(r["message"][:150], "\nconfidence:", r["confidence"])

In [ ]:
# Combined image + text, and an out-of-scope query (uncertainty handling)
r = bot.respond(text="which level is this on?", image=Image.open("data/images/baggage_claim/baggage_claim_005.jpg"))
print("fusion:", r["details"]["fusion_mode"], "->", r["record"]["name"], r["confidence"])
r = bot.respond(text="What is the weather like in Tokyo tomorrow?")
print("\nout-of-scope ->", r["message"][:120])

## 6. Deployment (Task 6)
Run the multimodal web interface locally:

```bash
streamlit run app/app.py
```

Optional Docker: `docker build -t airport-bot . && docker run -p 8501:8501 airport-bot`

## Optional online upgrades (documented, not required)
With internet access, CLIP and Whisper can replace the offline components:

In [ ]:
# OPTIONAL — requires internet + `pip install transformers openai-whisper`
# from transformers import CLIPModel, CLIPProcessor      # vision: CLIP embeddings + FAISS
# clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# import whisper                                          # speech: Whisper STT
# w = whisper.load_model("base"); print(w.transcribe("data/audio/q000.wav")["text"])
print("See report Section 4 for the upgrade discussion.")